# Repartition vs Coalesce: Full Shuffle vs Local Partition Merge

**Module**: 05_partitions_and_shuffle  
**Notebook**: 02_repartition_and_coalesce.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: The architectural difference between repartition() and coalesce(), when to use each, and how to avoid the small files problem.
- **Why does this topic exist?**: Using repartition when coalesce suffices triggers a massive, unnecessary network shuffle.
- **What problem does it solve?**: Slow writes and excessive network saturation.
- **Where is it used in real-world Data Engineering?**: Downsizing partition counts after filtering before writing to Parquet data lakes.

---

## 2. Architectural Comparison

```
COALESCE(2) [Local Merge - Narrow Transformation, No Shuffle]:
Worker 1: [P0] + [P1] ----> [Merged P0]
Worker 2: [P2] + [P3] ----> [Merged P1]
(No data travels across the network between workers!)

REPARTITION(2) [Round-Robin / Hash - Wide Transformation, Full Shuffle]:
Worker 1: [P0], [P1] ---\ /--- Network Shuffle ---> [New P0]
Worker 2: [P2], [P3] ---/ \--- Network Shuffle ---> [New P1]
(All records serialized and exchanged over network!)
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Repartition_Coalesce_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

## 3. Benchmarking Repartition vs Coalesce Execution Plans

In [ ]:
orders = spark.read.option("header", True).csv("/opt/spark-data/retail/orders.csv")

# Start with 8 partitions
base_df = orders.repartition(8)
print("Base partitions:", base_df.rdd.getNumPartitions())

print("\n=== COALESCE(2) Physical Plan (Notice: NO Exchange!) ===")
base_df.coalesce(2).explain()

print("\n=== REPARTITION(2) Physical Plan (Notice: Exchange Node!) ===")
base_df.repartition(2).explain()

---

## Experiment: Shuffle Write Bytes: Coalesce vs Repartition

- **1. What are we testing?**: Compare shuffle metrics between coalesce and repartition.
- **2. What do we expect?**: coalesce writes 0 shuffle bytes; repartition writes 100% of data to shuffle files.

In [ ]:
t0 = time.time()
base_df.coalesce(2).write.mode("overwrite").format("noop").save()
t_coal = time.time() - t0

t0 = time.time()
base_df.repartition(2).write.mode("overwrite").format("noop").save()
t_rep = time.time() - t0

print(f"Coalesce(2) time:    {t_coal:.3f}s (Zero shuffle)")
print(f"Repartition(2) time: {t_rep:.3f}s (Full network shuffle)")

**3. What happened?**: Coalesce completed significantly faster with zero shuffle overhead.  
**4. Why did it happen?**: Coalesce merged partition slices locally on each worker without network transfer.  
**5. Practical Engineering Lesson**: Always use coalesce() when reducing partition count after filters or before writes.

---

## Practical Exercises

### Exercise 1 (Beginner): Coalesce Single Output File
**Problem Statement**: Write sales.csv to Parquet using coalesce(1).  
**Input / Context**: sales.csv

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Hash Repartitioning by Key
**Problem Statement**: Repartition orders by customer_id into 4 partitions and check if rows with same customer reside in same partition.  
**Input / Context**: orders

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Detect Coalesce Partition Skew
**Problem Statement**: Create an uneven RDD, coalesce it, and inspect partition distribution using glom.  
**Input / Context**: glom

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Coalesce Single Output File):
```python
spark.read.option('header', True).csv('/opt/spark-data/sales.csv').coalesce(1).write.mode('overwrite').parquet('/opt/spark-data/warehouse/test_single_file')
```

#### Solution 2 (Hash Repartitioning by Key):
```python
p_df = orders.repartition(4,'customer_id')
print('Partitions:', p_df.rdd.getNumPartitions())
```

#### Solution 3 (Detect Coalesce Partition Skew):
```python
s = orders.coalesce(2).rdd.glom().map(len).collect()
print('Coalesced partition distribution:', s)
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **coalesce**: Merges locally on node; no shuffle; CAN ONLY DECREASE.
- **repartition**: Full network shuffle; increases or decreases; balances partition sizes.

### Essential Technical Terms
- `Exchange`: Shuffle node in plan.
- `Small Files Problem`: Too many tiny files written to storage.

### Common Engineering Mistakes
- **Using coalesce to increase partitions**: Fails silently; partition count remains unchanged.

### Interview Questions on This Topic
- **Q: When should you use repartition instead of coalesce?**
  - *A*: When increasing partition count, eliminating data skew, or partitioning explicitly by hash keys.

### What to Study Next
Open 03_shuffle_internals.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')